In [1]:
%pip install tdmclient

Note: you may need to restart the kernel to use updated packages.


# Setup

Imports, parameters and helper functions. **Set `ARENA_WIDTH_CM`** before running.

In [2]:

import cv2
import numpy as np

import Interface as ui          # Calibration, snapshot and display
import MotionController as mc   # Thymio movements (turn_to / go_forward)
import Filtering as kf


# ============================================================
# PARAMETERS
# ============================================================

ARENA_WIDTH_CM = 100       # To measure: distance (cm) between the outer
                            # corners of the left and right QR codes

GOAL_TOLERANCE_CM = 2.0     # Goal considered reached below this distance
SIMPLIFY_EPSILON_PX = 2.0   # Path simplification tolerance (map pixels)
MAX_STEP_CM = 10.0          # Maximum distance driven before measuring again
MIN_WAYPOINT_CM = 1.0       # Waypoints closer than this distance are skipped
FLUSH_FRAMES = 5            # Number of old frames discarded after a move


# ============================================================
# COORDINATE CONVERSIONS
# Map coordinates (pixels, y down)
# to world coordinates (cm, y up)
# ============================================================

def px_to_cm():
    """Calculate the conversion factor from pixels to centimeters."""

    if ARENA_WIDTH_CM is None:
        raise ValueError(
            "Set ARENA_WIDTH_CM: width of the arena between "
            "the outer corners of the QR codes (cm)."
        )

    return ARENA_WIDTH_CM / ui.MAP_WIDTH


def to_world(p):
    """Convert a map point (pixels) into world coordinates (cm)."""

    s = px_to_cm()

    return (
        p[0] * s,
        -p[1] * s
    )


def heading_world(theta):
    """
    Convert the camera heading into the robot's world heading.

    Camera convention: red -> green, with y pointing down.
    World convention: mathematical coordinates, with y pointing up.
    """

    return -(theta + ui.HEADING_OFFSET)


def distance(a, b):
    """Calculate the Euclidean distance between two points."""

    return float(
        np.hypot(
            b[0] - a[0],
            b[1] - a[1]
        )
    )


# ============================================================
# PATH PROCESSING
# ============================================================

def simplify(path):
    """
    Simplify an A* path (one point per pixel)
    by keeping only the main corners.
    """

    pts = np.array(path, dtype=np.float32).reshape(-1, 1, 2)

    simplified = cv2.approxPolyDP(
        pts,
        SIMPLIFY_EPSILON_PX,
        False
    )

    # float(): approxPolyDP gives numpy.float32, which tdmclient's client.sleep() refuses
    return [
        (float(p[0]), float(p[1]))
        for p in simplified.reshape(-1, 2)
    ]


def next_target(position, path):
    """
    Determine the next target point.

    Select the first waypoint sufficiently far from the robot.
    If the waypoint is farther than MAX_STEP_CM, create an
    intermediate target at the maximum allowed distance.
    """

    waypoints = [
        to_world(w)
        for w in simplify(path)
    ]

    waypoint = next(
        (
            w for w in waypoints[1:]
            if distance(position, w) > MIN_WAYPOINT_CM
        ),
        waypoints[-1]
    )

    d = distance(position, waypoint)

    if d <= MAX_STEP_CM:
        return waypoint

    r = MAX_STEP_CM / d

    return (
        position[0] + r * (waypoint[0] - position[0]),
        position[1] + r * (waypoint[1] - position[1])
    )


# ============================================================
# DISPLAY
# ============================================================

def show(frame, state, H, H_inv):
    """
    Display the camera image and the rectified map.

    Returns the key pressed by the user.
    """

    # Display the original camera image
    camera_view = ui.draw_camera(
        frame.copy(),
        state,
        H_inv,
        stale=False
    )

    cv2.imshow(ui.WINDOW_CAMERA, camera_view)

    # Apply perspective transformation to obtain the map
    rectified = cv2.warpPerspective(
        frame,
        H,
        (ui.MAP_WIDTH, ui.MAP_HEIGHT)
    )

    # Display the rectified map
    map_view = ui.draw_map(
        rectified,
        state,
        stale=False
    )

    cv2.imshow(ui.WINDOW_MAP, map_view)

    return cv2.waitKey(1) & 0xFF


# Run

Calibration, then loop: snapshot -> display -> move to the next point -> new snapshot.

Keys: `q` in the camera window to stop (checked between two moves).

Restart the kernel before running if `tdmclient.notebook.start()` was run (the Thymio would be locked: "busy").

In [ ]:
px_to_cm()   # check ARENA_WIDTH_CM before starting

cap = cv2.VideoCapture(ui.CAMERA_INDEX, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, ui.CAMERA_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, ui.CAMERA_HEIGHT)

try:
    if not cap.isOpened():
        raise RuntimeError("Unable to open the camera")
    print(f"Camera: {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))} x {int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")

    # 1. Camera calibration (once)
    print("Calibration... (show the 4 QR codes, q to quit)")
    calibration = ui.calibrate_camera(cap)
    if calibration is None:
        raise RuntimeError("Calibration aborted ('q' pressed)")
    H, H_inv = calibration
    print("Calibration done.")

    # 2. Main loop: snapshot -> one segment with run() -> new snapshot
    while True:
        for _ in range(FLUSH_FRAMES):    # drop the frames taken during the previous move
            cap.read()
        ret, frame = cap.read()
        if not ret:
            raise RuntimeError("Unable to read a frame from the camera")

        state = ui.take_snapshot(frame)  # x, y, theta, path from fetch_status_cv
        if show(frame, state, H, H_inv) == ord('q'):
            print("Stopped by the user.")
            break

        if state["thymio"] is None or state["path"] is None:
            print("Snapshot not usable:", state["error"] or "no path found")
            await mc.client.sleep(0.2)   # the motors are already stopped by run()
            continue

        position = to_world(state["thymio"])
        if distance(position, to_world(state["path"][-1])) < GOAL_TOLERANCE_CM:
            print("Goal reached!")
            break

        # data = [x0, y0, theta0, path] for run() (MotionController):
        # start pose measured by the camera, path = next segment only (then new snapshot)
        #data = [position[0], position[1], heading_world(state["theta"]),[next_target(position, state["path"])]]
        
        k_state = [position[0]/100, position[1]/100, heading_world(state["theta"])]
        speed_left = motor.left.speed # a corriger
        speed_right = motor.right.speed # a corriger
        dt = 0.05
        camera_pose = [position[0]/100, position[1]/100, heading_world(state["theta"])]
        P = kf.R.copy
        
        data = kf.kalman(k_state, P, speed_left, speed_right, dt, camera_pose)
        await mc.run(data)   # run -> follow_path -> move_to; locks the Thymio, stops the motors at the end

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Stopped: motors off, camera closed.")